# Autoship Delay in Onboarding — Data Analysis

This notebook builds up, CTE by CTE, the two queries that actually run in `power_analysis.ipynb`: **Step 0**'s `validate_platform_match` and **Step 1**'s `baseline_query`. Each step below adds exactly one CTE from those two queries and re-runs, so the final step in each part reproduces the exact query and numbers in `power_analysis.ipynb`.

In [1]:
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

## Part A — `validate_platform_match` (Step 0)

Built up using the same parameters as Step 0's April 2026 call: `cohort_start='2026-04-01'`, `cohort_end='2026-05-01'`, `events_start='2026-03-31'`, `events_end='2026-05-02'`.

### A1 — the `cohort` CTE alone

Just the Style Profile population: Womens/Mens clients who completed Style Profile in April 2026.

In [2]:
query("""--sql
SELECT client_id, style_profile_completed_ts, business_line
FROM curated.client
WHERE style_profile_completed_ts IS NOT NULL
  AND business_line IN ('Womens', 'Mens')
  AND DATE(style_profile_completed_ts) >= DATE '2026-04-01'
  AND DATE(style_profile_completed_ts) < DATE '2026-05-01'
ORDER BY client_id
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,style_profile_completed_ts,business_line
0,353,2026-04-03 23:02:34.088,Womens
1,3005971,2026-04-01 02:05:19.106,Womens
2,3012824,2026-04-03 03:00:36.789,Womens
3,3013387,2026-04-13 20:43:13.247,Womens
4,3014245,2026-04-10 16:32:02.067,Womens


In [3]:
query("""--sql
SELECT COUNT(*) AS n_cohort
FROM curated.client
WHERE style_profile_completed_ts IS NOT NULL
  AND business_line IN ('Womens', 'Mens')
  AND DATE(style_profile_completed_ts) >= DATE '2026-04-01'
  AND DATE(style_profile_completed_ts) < DATE '2026-05-01'
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_cohort
0,139646


139,646 — matches `n_cohort` in Step 0's April result. This is the denominator the platform match rate is measured against.

### A2 — the `sp_events` CTE alone

The candidate events: rows in `curated.product_tracking_events` tied to the Style Profile flow (by `schema`, or a `screen_view` on the known screen names), in the same date window as the cohort. Not yet joined to any specific client in `cohort`.

In [4]:
query("""--sql
SELECT COUNT(*) AS n_candidate_events, COUNT(DISTINCT client_id) AS n_distinct_clients
FROM curated.product_tracking_events
WHERE client_id IS NOT NULL
  AND date_in_utc >= DATE '2026-03-31'
  AND date_in_utc < DATE '2026-05-02'
  AND (
        schema IN ('style_profile_select', 'style_profile_view')
        OR (type = 'screen_view' AND screen_view_name IN ('style_profile', 'client_style_profile', 'stylefile_onboarding'))
      )
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_candidate_events,n_distinct_clients
0,18982927,391523


~19M candidate event rows across ~391K distinct clients in this window — far more clients than the 139,646 in `cohort`, since this table isn't restricted to April's cohort alone (it's just filtered by date and event type, exactly as in the real query). The next step is where `cohort` and `sp_events` actually meet.

### A3 — the `matched` CTE, before collapsing to the nearest event

`cohort` LEFT JOIN `sp_events` on `client_id`, within the `[-6h, +1h]` window, ranked by `ROW_NUMBER()` per client — ordered by closeness in time. Shown here for 3 real clients, uncollapsed, so the ranking is visible.

In [5]:
query("""--sql
WITH cohort AS (
    SELECT client_id, style_profile_completed_ts, business_line
    FROM curated.client
    WHERE style_profile_completed_ts IS NOT NULL
      AND business_line IN ('Womens', 'Mens')
      AND DATE(style_profile_completed_ts) >= DATE '2026-04-01'
      AND DATE(style_profile_completed_ts) < DATE '2026-05-01'
),
sp_events AS (
    SELECT client_id, platform, datetime_in_utc
    FROM curated.product_tracking_events
    WHERE client_id IS NOT NULL
      AND date_in_utc >= DATE '2026-03-31'
      AND date_in_utc < DATE '2026-05-02'
      AND (
            schema IN ('style_profile_select', 'style_profile_view')
            OR (type = 'screen_view' AND screen_view_name IN ('style_profile', 'client_style_profile', 'stylefile_onboarding'))
          )
),
matched AS (
    SELECT
        c.client_id,
        e.platform,
        e.datetime_in_utc,
        c.style_profile_completed_ts,
        ROW_NUMBER() OVER (
            PARTITION BY c.client_id
            ORDER BY ABS(date_diff('second', e.datetime_in_utc, c.style_profile_completed_ts))
        ) AS rn
    FROM cohort c
    LEFT JOIN sp_events e
        ON e.client_id = c.client_id
        AND e.datetime_in_utc BETWEEN c.style_profile_completed_ts - INTERVAL '6' HOUR
                                   AND c.style_profile_completed_ts + INTERVAL '1' HOUR
)
SELECT client_id, style_profile_completed_ts, datetime_in_utc, platform, rn
FROM matched
WHERE client_id IN (3005971, 3012824, 3013387) AND rn <= 3
ORDER BY client_id, rn
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,style_profile_completed_ts,datetime_in_utc,platform,rn
0,3005971,2026-04-01 02:05:19.106,2026-04-01 02:05:18.412,web,1
1,3005971,2026-04-01 02:05:19.106,2026-04-01 02:05:24.377,web,2
2,3005971,2026-04-01 02:05:19.106,2026-04-01 02:05:12.360,web,3
3,3012824,2026-04-03 03:00:36.789,2026-04-03 03:00:36.245,web,1
4,3012824,2026-04-03 03:00:36.789,2026-04-03 03:00:42.142,web,2
5,3012824,2026-04-03 03:00:36.789,2026-04-03 03:00:29.999,web,3
6,3013387,2026-04-13 20:43:13.247,2026-04-13 20:43:12.776,web,1
7,3013387,2026-04-13 20:43:13.247,2026-04-13 20:43:07.350,web,2
8,3013387,2026-04-13 20:43:13.247,2026-04-13 20:43:07.367,web,3


Each client has multiple candidate events within the window; `rn = 1` is the one within a second of `style_profile_completed_ts` — that's the row Step 0 keeps. Rows `rn = 2, 3` are the runner-up candidates, a few seconds off. Only `rn = 1` survives into the final query.

### A4 — the full `validate_platform_match` query

Add the `WHERE rn = 1` collapse and the aggregation (`n_cohort`, `n_matched`, `n_web`, `n_ios`, `n_other_platform`) — this is the exact query in Step 0.

In [6]:
query("""--sql
WITH cohort AS (
    SELECT client_id, style_profile_completed_ts, business_line
    FROM curated.client
    WHERE style_profile_completed_ts IS NOT NULL
      AND business_line IN ('Womens', 'Mens')
      AND DATE(style_profile_completed_ts) >= DATE '2026-04-01'
      AND DATE(style_profile_completed_ts) < DATE '2026-05-01'
),
sp_events AS (
    SELECT client_id, platform, datetime_in_utc
    FROM curated.product_tracking_events
    WHERE client_id IS NOT NULL
      AND date_in_utc >= DATE '2026-03-31'
      AND date_in_utc < DATE '2026-05-02'
      AND (
            schema IN ('style_profile_select', 'style_profile_view')
            OR (type = 'screen_view' AND screen_view_name IN ('style_profile', 'client_style_profile', 'stylefile_onboarding'))
          )
),
matched AS (
    SELECT
        c.client_id,
        e.platform,
        ROW_NUMBER() OVER (
            PARTITION BY c.client_id
            ORDER BY ABS(date_diff('second', e.datetime_in_utc, c.style_profile_completed_ts))
        ) AS rn
    FROM cohort c
    LEFT JOIN sp_events e
        ON e.client_id = c.client_id
        AND e.datetime_in_utc BETWEEN c.style_profile_completed_ts - INTERVAL '6' HOUR
                                   AND c.style_profile_completed_ts + INTERVAL '1' HOUR
)
SELECT
    (SELECT count(*) FROM cohort) as n_cohort,
    count(*) FILTER (WHERE rn=1 AND platform IS NOT NULL) as n_matched,
    count(*) FILTER (WHERE rn=1 AND platform='web') as n_web,
    count(*) FILTER (WHERE rn=1 AND platform='iOS') as n_ios,
    count(*) FILTER (WHERE rn=1 AND platform NOT IN ('web','iOS')) as n_other_platform
FROM matched
WHERE rn = 1
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_cohort,n_matched,n_web,n_ios,n_other_platform
0,139646,139636,127297,12339,0


Matches Step 0's April result exactly: 139,636 / 139,646 matched, 127,297 web / 12,339 iOS.

## Part B — `baseline_query` (Step 1)

Built up using Step 1's actual parameters: `COHORT_START = '2025-10-01'`, `MATURATION_DAYS = 60`. Reuses the same `cohort` → `sp_events` → `matched` mechanics from Part A, on a different cohort window (multi-month, with the maturation cutoff), then adds the web filter, the conversion join, and the monthly grouping.

### B1 — the `cohort` CTE alone, with the maturation cutoff

Same shape as A1, but spanning the full `COHORT_START` window and requiring `style_profile_completed_ts <= CURRENT_DATE - 60 days` — i.e., only cohorts old enough for a fix to have had time to resolve.

In [7]:
query("""--sql
WITH cohort AS (
    SELECT client_id, style_profile_completed_ts, business_line
    FROM curated.client
    WHERE style_profile_completed_ts IS NOT NULL
      AND business_line IN ('Womens', 'Mens')
      AND DATE(style_profile_completed_ts) >= DATE '2025-10-01'
      AND style_profile_completed_ts <= CURRENT_DATE - INTERVAL '60' DAY
)
SELECT DATE_TRUNC('month', style_profile_completed_ts) AS month, COUNT(*) AS n_cohort
FROM cohort
GROUP BY 1
ORDER BY 1 DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,n_cohort
0,2026-05-01 00:00:00.000,63164
1,2026-04-01 00:00:00.000,139646
2,2026-03-01 00:00:00.000,180686
3,2026-02-01 00:00:00.000,167101
4,2026-01-01 00:00:00.000,184778
5,2025-12-01 00:00:00.000,126128
6,2025-11-01 00:00:00.000,132850
7,2025-10-01 00:00:00.000,152252


All-platform Style Profile completions per month, already excluding anything younger than 60 days (no June/July rows here). May is partial (63,164) since only the first ~15 days of May had passed the 60-day mark as of the run date.

### B2 — add `matched` + `web_cohort`: the platform filter

Same `matched` mechanics as Part A (client_id + platform join, `[-6h, +1h]` window, same event-type filter), then keep only `rn = 1 AND platform = 'web'`.

In [8]:
query("""--sql
WITH cohort AS (
    SELECT client_id, style_profile_completed_ts, business_line
    FROM curated.client
    WHERE style_profile_completed_ts IS NOT NULL
      AND business_line IN ('Womens', 'Mens')
      AND DATE(style_profile_completed_ts) >= DATE '2025-10-01'
      AND style_profile_completed_ts <= CURRENT_DATE - INTERVAL '60' DAY
),
sp_events AS (
    SELECT client_id, platform, datetime_in_utc
    FROM curated.product_tracking_events
    WHERE client_id IS NOT NULL
      AND date_in_utc >= DATE '2025-09-30'
      AND date_in_utc < DATE '2026-06-02'
      AND (
            schema IN ('style_profile_select', 'style_profile_view')
            OR (type = 'screen_view' AND screen_view_name IN ('style_profile', 'client_style_profile', 'stylefile_onboarding'))
          )
),
matched AS (
    SELECT
        c.client_id,
        c.style_profile_completed_ts,
        e.platform,
        ROW_NUMBER() OVER (
            PARTITION BY c.client_id
            ORDER BY ABS(date_diff('second', e.datetime_in_utc, c.style_profile_completed_ts))
        ) AS rn
    FROM cohort c
    LEFT JOIN sp_events e
        ON e.client_id = c.client_id
        AND e.datetime_in_utc BETWEEN c.style_profile_completed_ts - INTERVAL '6' HOUR
                                   AND c.style_profile_completed_ts + INTERVAL '1' HOUR
),
web_cohort AS (
    SELECT client_id, style_profile_completed_ts
    FROM matched
    WHERE rn = 1 AND platform = 'web'
)
SELECT DATE_TRUNC('month', style_profile_completed_ts) AS month, COUNT(*) AS n_web_cohort
FROM web_cohort
GROUP BY 1
ORDER BY 1 DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,n_web_cohort
0,2026-05-01 00:00:00.000,57162
1,2026-04-01 00:00:00.000,127297
2,2026-03-01 00:00:00.000,165990
3,2026-02-01 00:00:00.000,153159
4,2026-01-01 00:00:00.000,167817
5,2025-12-01 00:00:00.000,114499
6,2025-11-01 00:00:00.000,120381
7,2025-10-01 00:00:00.000,138113


Every month drops relative to B1 (e.g., April: 139,646 → 127,297) — the app clients being filtered out. These numbers are exactly `n_style_profile_completions_web` in Step 1's final table.

### B3 — add `joined`: the conversion flag

Left join `web_cohort` to `curated.client_first_conversion`, flagging `cancellation_adjusted_first_fix_conversion` only if it happened within `MATURATION_DAYS` of completion.

In [9]:
query("""--sql
WITH cohort AS (
    SELECT client_id, style_profile_completed_ts, business_line
    FROM curated.client
    WHERE style_profile_completed_ts IS NOT NULL
      AND business_line IN ('Womens', 'Mens')
      AND DATE(style_profile_completed_ts) >= DATE '2025-10-01'
      AND style_profile_completed_ts <= CURRENT_DATE - INTERVAL '60' DAY
),
sp_events AS (
    SELECT client_id, platform, datetime_in_utc
    FROM curated.product_tracking_events
    WHERE client_id IS NOT NULL
      AND date_in_utc >= DATE '2025-09-30'
      AND date_in_utc < DATE '2026-06-02'
      AND (
            schema IN ('style_profile_select', 'style_profile_view')
            OR (type = 'screen_view' AND screen_view_name IN ('style_profile', 'client_style_profile', 'stylefile_onboarding'))
          )
),
matched AS (
    SELECT
        c.client_id,
        c.style_profile_completed_ts,
        e.platform,
        ROW_NUMBER() OVER (
            PARTITION BY c.client_id
            ORDER BY ABS(date_diff('second', e.datetime_in_utc, c.style_profile_completed_ts))
        ) AS rn
    FROM cohort c
    LEFT JOIN sp_events e
        ON e.client_id = c.client_id
        AND e.datetime_in_utc BETWEEN c.style_profile_completed_ts - INTERVAL '6' HOUR
                                   AND c.style_profile_completed_ts + INTERVAL '1' HOUR
),
web_cohort AS (
    SELECT client_id, style_profile_completed_ts
    FROM matched
    WHERE rn = 1 AND platform = 'web'
),
joined AS (
    SELECT
        w.client_id,
        w.style_profile_completed_ts,
        DATE_TRUNC('month', w.style_profile_completed_ts) AS month,
        CASE WHEN v.cancellation_adjusted_first_fix_demand_ts IS NOT NULL
              AND v.cancellation_adjusted_first_fix_demand_ts <= w.style_profile_completed_ts + INTERVAL '60' DAY
             THEN 1 ELSE 0 END AS cancellation_adjusted_first_fix_conversion
    FROM web_cohort w
    LEFT JOIN curated.client_first_conversion v ON w.client_id = v.client_id
)
SELECT
    month,
    COUNT(*) AS n_web_cohort,
    SUM(cancellation_adjusted_first_fix_conversion) AS n_cancellation_adjusted_first_fix,
    CAST(SUM(cancellation_adjusted_first_fix_conversion) AS DOUBLE) / COUNT(*) AS first_fix_conversion_rate
FROM joined
GROUP BY month
ORDER BY month DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,n_web_cohort,n_cancellation_adjusted_first_fix,first_fix_conversion_rate
0,2026-05-01 00:00:00.000,57162,15671,0.274151
1,2026-04-01 00:00:00.000,127297,34253,0.269079
2,2026-03-01 00:00:00.000,165990,43484,0.261968
3,2026-02-01 00:00:00.000,153159,36660,0.239359
4,2026-01-01 00:00:00.000,167817,38732,0.230799
5,2025-12-01 00:00:00.000,114499,25052,0.218797
6,2025-11-01 00:00:00.000,120381,26969,0.224030
7,2025-10-01 00:00:00.000,138113,33590,0.243207


This is already the baseline rate used in Step 1/2/3 (April: 26.9%) — the only thing missing is the daily-volume column, which needs one more CTE.

### B4 — the full `baseline_query`

Add `month_days` (distinct calendar days observed per month) and the final `SELECT`, which turns the monthly count into a per-day rate. This is the exact query in Step 1.

In [10]:
baseline_query = """--sql
WITH cohort AS (
    SELECT client_id, style_profile_completed_ts, business_line
    FROM curated.client
    WHERE style_profile_completed_ts IS NOT NULL
      AND business_line IN ('Womens', 'Mens')
      AND DATE(style_profile_completed_ts) >= DATE '2025-10-01'
      AND style_profile_completed_ts <= CURRENT_DATE - INTERVAL '60' DAY
),
sp_events AS (
    SELECT client_id, platform, datetime_in_utc
    FROM curated.product_tracking_events
    WHERE client_id IS NOT NULL
      AND date_in_utc >= DATE '2025-09-30'
      AND date_in_utc < DATE '2026-06-02'
      AND (
            schema IN ('style_profile_select', 'style_profile_view')
            OR (type = 'screen_view' AND screen_view_name IN ('style_profile', 'client_style_profile', 'stylefile_onboarding'))
          )
),
matched AS (
    SELECT
        c.client_id,
        c.style_profile_completed_ts,
        e.platform,
        ROW_NUMBER() OVER (
            PARTITION BY c.client_id
            ORDER BY ABS(date_diff('second', e.datetime_in_utc, c.style_profile_completed_ts))
        ) AS rn
    FROM cohort c
    LEFT JOIN sp_events e
        ON e.client_id = c.client_id
        AND e.datetime_in_utc BETWEEN c.style_profile_completed_ts - INTERVAL '6' HOUR
                                   AND c.style_profile_completed_ts + INTERVAL '1' HOUR
),
web_cohort AS (
    SELECT client_id, style_profile_completed_ts
    FROM matched
    WHERE rn = 1 AND platform = 'web'
),
joined AS (
    SELECT
        w.client_id,
        w.style_profile_completed_ts,
        DATE_TRUNC('month', w.style_profile_completed_ts) AS month,
        CASE WHEN v.cancellation_adjusted_first_fix_demand_ts IS NOT NULL
              AND v.cancellation_adjusted_first_fix_demand_ts <= w.style_profile_completed_ts + INTERVAL '60' DAY
             THEN 1 ELSE 0 END AS cancellation_adjusted_first_fix_conversion
    FROM web_cohort w
    LEFT JOIN curated.client_first_conversion v ON w.client_id = v.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT DATE(style_profile_completed_ts)) AS days_observed
    FROM joined
    GROUP BY month
)
SELECT
    j.month,
    md.days_observed,
    COUNT(*) AS n_style_profile_completions_web,
    SUM(cancellation_adjusted_first_fix_conversion) AS n_cancellation_adjusted_first_fix,
    CAST(SUM(cancellation_adjusted_first_fix_conversion) AS DOUBLE) / COUNT(*) AS first_fix_conversion_rate,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS style_profile_completions_per_day
FROM joined j
JOIN month_days md ON j.month = md.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

query(baseline_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_style_profile_completions_web,n_cancellation_adjusted_first_fix,first_fix_conversion_rate,style_profile_completions_per_day
0,2026-05-01 00:00:00.000,15,57162,15671,0.274151,3810.8
1,2026-04-01 00:00:00.000,30,127297,34253,0.269079,4243.2
2,2026-03-01 00:00:00.000,31,165990,43484,0.261968,5354.5
3,2026-02-01 00:00:00.000,28,153159,36660,0.239359,5470.0
4,2026-01-01 00:00:00.000,31,167817,38732,0.230799,5413.5
5,2025-12-01 00:00:00.000,31,114499,25052,0.218797,3693.5
6,2025-11-01 00:00:00.000,30,120381,26969,0.224030,4012.7
7,2025-10-01 00:00:00.000,31,138113,33590,0.243207,4455.3


Matches Step 1 exactly, including the April 2026 reference row used for the baseline: 26.9% conversion, 4,243.2 completions/day.